# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset ([Croissant schema on sen.science](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)) using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

The FAIR^2 dataset provides ordered logistic regression outputs, including socio-demographic predictors of indigenous and modern knowledge adoption among pastoralist households in Northern Kenya.

### Dataset Source

The dataset schema is defined as a [Croissant JSON-LD document](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Ensure mlcroissant is installed
!pip install -q mlcroissant

## 1. Data Loading

Load dataset metadata and records from the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {getattr(metadata, 'name', '')}\n{getattr(metadata, 'description', '')}")

### Dataset Metadata Highlights
- **Identifier**: 10.71728/senscience.y7m0-f273
- **Published**: 2026-07-29
- **Spatial Coverage**: Samburu, Isiolo, Marsabit counties, Northern Kenya
- **Keywords**: adoption predictors, climate adaptation, extension services, gender inclusion, indigenous knowledge
- **License**: [Open Data Commons BY 1.0](https://opendatacommons.org/licenses/by/1-0/)


## 2. Data Overview

Explore the available record sets, their `@id` values, and fields. In the Croissant data model, record sets group entities (e.g., tables or structured outputs), and fields correspond to the columns/attributes in those tables.

We start by listing all record sets (`cr:RecordSet`) and display their IDs and columns.

In [ ]:
# List available record sets with their @id and name/title.
print("Available record sets in this dataset:")
for record_set in dataset.record_sets:
    print(f" - @id: {record_set.id} | name: {getattr(record_set, 'name', '')}")

print("\nDisplaying columns and their @id for each record set:")
for record_set in dataset.record_sets:
    print(f"\nRecordSet: @id: {record_set.id} | name: {getattr(record_set, 'name', '')}")
    if hasattr(record_set, 'columns'):
        for col in record_set.columns:
            print(f"   - column @id: {col.id} | name: {getattr(col, 'name', '')}")
    else:
        print("   (No columns found)")

## 3. Data Extraction

Load data from the primary record set(s) into pandas DataFrames for further analysis. Reference all entities by their `@id`.

We'll extract all data for inspection and identify main record set(s) for analysis.

In [ ]:
# Gather the @id values of all record sets
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
# Load data for each record set by its @id
for rec_set_id in record_set_ids:
    # Records can be a generator; convert to list first, then DataFrame
    records = list(dataset.records(record_set=rec_set_id))
    df = pd.DataFrame(records)
    dataframes[rec_set_id] = df
    print(f"Loaded {len(df)} rows from record set @id: {rec_set_id}")
    if not df.empty:
        print(f"Columns: {df.columns.tolist()}")
    print()

# For demonstration, select the first available (non-empty) record set for in-depth exploration
selected_record_set_id = None
for rec_set_id, df in dataframes.items():
    if not df.empty:
        selected_record_set_id = rec_set_id
        break
if selected_record_set_id:
    print(f"Selected record set for analysis: {selected_record_set_id}")
    print(dataframes[selected_record_set_id].head())
else:
    print("No non-empty record sets found.")

## 4. Exploratory Data Analysis (EDA)

Let's demonstrate standard EDA workflows:
- Filtering by a numeric column (by `@id`)
- Normalizing this numeric column within a filtered subset
- Grouping by a categorical or grouping field (by `@id`) and summarizing

_You can update the specified field and group IDs from the overview above as appropriate to your dataset!_

In [ ]:
# Choose a numeric field by its @id (update this variable for your analysis!)
example_numeric_field_id = None
example_group_field_id = None
df = dataframes[selected_record_set_id]

# Optionally auto-detect a numeric field by pandas dtype (for demonstration)
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]) and not df[col].isnull().all():
        example_numeric_field_id = col
        break
# Auto-detect group field (first non-numeric, non-null column)
for col in df.columns:
    if not pd.api.types.is_numeric_dtype(df[col]) and not df[col].isnull().all():
        example_group_field_id = col
        break

print(f"Using numeric field for filter/normalization: {example_numeric_field_id}")
print(f"Using grouping field: {example_group_field_id}")

# EDA: filter for values above a threshold
if example_numeric_field_id:
    threshold = df[example_numeric_field_id].mean()  # example dynamic threshold
    filtered_df = df[df[example_numeric_field_id] > threshold]
    print(f"Filtered records where {example_numeric_field_id} > {threshold:.2f} (mean):")
    print(filtered_df.head())

    # Normalize numeric field (z-score)
    filtered_df[example_numeric_field_id + '_normalized'] = (
        (filtered_df[example_numeric_field_id] - filtered_df[example_numeric_field_id].mean()) /
        filtered_df[example_numeric_field_id].std()
    )
    print(f"\nNormalized {example_numeric_field_id} for filtered records:")
    print(filtered_df[[example_numeric_field_id, example_numeric_field_id + '_normalized']].head())

    # Group by 
    if example_group_field_id and example_group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(example_group_field_id)[example_numeric_field_id].mean().reset_index()
        print(f"\nGrouped by {example_group_field_id}, mean {example_numeric_field_id}:")
        print(grouped.head())
else:
    print("No numeric field found to demonstrate filtering and normalization.")

## 5. Visualization

Visualize the distribution of the selected numeric field and the relationship between groupings (if any), using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[example_numeric_field_id].dropna(), kde=True, bins=30, color='skyblue')
    plt.title(f"Distribution of: {example_numeric_field_id}")
    plt.xlabel(example_numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if example_group_field_id:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[example_group_field_id], y=df[example_numeric_field_id])
        plt.title(f"{example_numeric_field_id} grouped by {example_group_field_id}")
        plt.xlabel(example_group_field_id)
        plt.ylabel(example_numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

This notebook demonstrated how to:
- Load FAIR^2 dataset metadata and data from a Croissant schema using `mlcroissant` (referencing entities by their `@id` fields)
- Explore available record sets, their columns (fields), and extract their content as pandas DataFrames
- Preview, filter, normalize, and group data using EDA tools
- Visualize distributions and relationships in the dataset

**Next steps:**
- Interpret logistic regression outputs for policy and intervention analysis
- Integrate additional record sets or external context if relevant

For more about the Croissant standard, see the [MLCommons documentation](https://mlcommons.org/croissant/). For FAIR^2 dataset context and domain implications, refer to the project [sen.science portal](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).